# 06 · SVD del texto, más modelos, stacking y reglas

El modelo del `05_errores_encodings.ipynb` (LightGBM + XGBoost con los encodings cruzados) llega a **0,9443 en la validación** y **0,94590 en Kaggle**. Este notebook prueba, del lado de los **modelos**, lo que todavía no se había probado:

1. **Fotos compartidas** entre publicaciones (para vincular test con train).
2. **CatBoost con vendedor y categoría como categóricas**, que arma solo sus *target encodings* y combinaciones.
3. **SVD del texto:** el TF-IDF resumido en unas decenas de columnas numéricas. **Es lo que funcionó.**
4. Otros modelos para el ensamble: **Extra Trees, Random Forest, modelo lineal**, y modelos que ven **distintas columnas**.
5. **Stacking:** un segundo modelo que aprende a combinar a los demás.
6. **Reglas a mano.**
7. **Modelo final y submission.**

Como en el 05, cada idea se compara contra una referencia **partición por partición**. Se acepta si mejora más que el ruido (~0,001) y en casi todas las particiones.

Todo el notebook tarda **casi 2 horas**. Que la compu no se suspenda y, con 7,7 GB de RAM, conviene cerrar otros programas.

## 0. Datos, features y piezas

Las piezas ahora incluyen los 3 encodings cruzados del 05 (`cruzados=True`). Tarda unos 13 minutos.

In [1]:
import re, time
import warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from itertools import combinations
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from features import (cargar_datos, armar_tabla, calcular_piezas, armar_conjunto, texto_combinado,
                      componentes_svd, CAT, g)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

todo, y = cargar_datos()
N_TRAIN = len(y)
X_todo = armar_tabla(todo)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
PARTICIONES = list(cv.split(np.zeros(N_TRAIN), y)) + [(np.arange(N_TRAIN), np.arange(N_TRAIN, len(todo)))]

inicio = time.time()
piezas = [calcular_piezas(todo, y, idx_a, idx_b, caracteres=True, cruzados=True) for idx_a, idx_b in PARTICIONES]
print(X_todo.shape, f"piezas listas en {(time.time() - inicio) / 60:.1f} minutos")

(100000, 92) piezas listas en 8.8 minutos


Funciones para evaluar (como en el 05):

- `oof(X, modelo, params)`: probabilidad *out of fold* de un modelo entrenado con las columnas de `X` + las piezas. Con `columnas=` se puede elegir un subconjunto.
- `comparar(nombre, p, ref)`: accuracy y diferencia partición por partición contra una referencia.

In [2]:
PARAMS_LGBM = dict(n_estimators=1500, learning_rate=0.02, num_leaves=63, colsample_bytree=0.5, random_state=42, verbose=-1)
PARAMS_XGB  = dict(n_estimators=1500, learning_rate=0.02, max_depth=8, subsample=0.8, colsample_bytree=0.5,
                   min_child_weight=2, tree_method="hist", enable_categorical=True, random_state=42, n_jobs=8)

def oof(X, modelo=LGBMClassifier, params=PARAMS_LGBM, preparar=None, columnas=None, **fit_kwargs):
    p = np.zeros(N_TRAIN)
    for f, (idx_a, idx_b) in enumerate(PARTICIONES[:5]):
        A, B = armar_conjunto(X, piezas[f], idx_a, idx_b)
        if columnas is not None:
            A, B = A[columnas], B[columnas]
        if preparar is not None:
            A, B = preparar(A), preparar(B)
        p[idx_b] = modelo(**params).fit(A, y[idx_a], **fit_kwargs).predict_proba(B)[:, 1]
    return p

def accuracy_por_particion(p):
    return np.array([accuracy_score(y[idx_b], p[idx_b] > 0.5) for _, idx_b in PARTICIONES[:5]])

def desacuerdo(p, q):
    return ((p > 0.5) != (q > 0.5)).mean()

resultados = []
def comparar(nombre, p, ref):
    notas, notas_ref = accuracy_por_particion(p), accuracy_por_particion(ref)
    dif = notas - notas_ref
    resultados.append({"experimento": nombre, "accuracy": round(notas.mean(), 4), "vs. referencia": round(dif.mean(), 4),
                       "mejora en": f"{(dif > 0).sum()}/5", "diferencia por partición": " ".join(f"{d:+.4f}" for d in dif)})
    print(f"{nombre:60s} {notas.mean():.4f}  ({dif.mean():+.4f}, mejora en {(dif > 0).sum()}/5)")

Referencia: el **modelo 05**, con LightGBM y XGBoost por separado y su promedio. Tarda unos 8 minutos.

In [3]:
p_lgbm = oof(X_todo)
p_xgb  = oof(X_todo, XGBClassifier, PARAMS_XGB)
p_05 = (p_lgbm + p_xgb) / 2
print(f"LightGBM {accuracy_por_particion(p_lgbm).mean():.4f} | XGBoost {accuracy_por_particion(p_xgb).mean():.4f} | "
      f"modelo 05 (promedio) {accuracy_por_particion(p_05).mean():.4f}")

LightGBM 0.9444 | XGBoost 0.9440 | modelo 05 (promedio) 0.9443


## 1. ¿Hay publicaciones que comparten fotos?

Si una publicación de test usara **la misma foto** que una de train, sería probablemente el mismo producto republicado, y su respuesta se conocería. También se mira si el número `MLA...` que aparece dentro del id de las fotos o de la descripción apunta a **otra** publicación del dataset.

In [4]:
fotos = todo["pictures"].apply(lambda L: [p["id"] for p in L])
tabla_fotos = pd.DataFrame({"fila": np.repeat(np.arange(len(todo)), fotos.apply(len)),
                            "foto": np.concatenate(fotos.values)}).drop_duplicates()
compartidas = tabla_fotos[tabla_fotos.groupby("foto")["fila"].transform("nunique") > 1]
print("publicaciones que comparten alguna foto con otra:", compartidas["fila"].nunique(), "de", len(todo))

id_num = todo["id"].str[3:]
existentes = set(id_num)
def apunta_a_otra(columna):
    numeros = columna.apply(lambda L: {m for x in L for m in re.findall(r"MLA(\d+)", str(x))})
    return sum(len({m for m in s if m != i and m in existentes}) > 0 for s, i in zip(numeros, id_num))
print("fotos que nombran a OTRA publicación del dataset:", apunta_a_otra(fotos))
print("descripciones que nombran a OTRA publicación del dataset:", apunta_a_otra(todo["descriptions"]))

publicaciones que comparten alguna foto con otra: 6 de 100000


fotos que nombran a OTRA publicación del dataset: 0


descripciones que nombran a OTRA publicación del dataset: 0


**Lectura:** prácticamente ninguna publicación comparte fotos y ningún id apunta a otra publicación del dataset: los ids están anonimizados. No hay forma de vincular test con train por este lado.

## 2. CatBoost con vendedor y categoría como categóricas

En el 04, CatBoost usaba solo las categóricas de pocos valores. Acá recibe también **`seller_id`, `category_id` y la primera palabra del título** en crudo. CatBoost hace dos cosas con ellas:

- Calcula su propio *target encoding* **"ordenado"**: ordena las filas al azar y, para cada fila, usa solo las filas **anteriores**. Así nunca usa la respuesta de la propia fila.
- Prueba **combinaciones** de categóricas (`max_ctr_complexity=2`: de a pares), que es lo que en el 05 se hizo a mano con los encodings cruzados.

Tarda unos 22 minutos.

In [5]:
palabra1 = todo["title"].fillna("").str.lower().str.extract(r"([a-záéíóúñ]+)", expand=False).fillna("")
X_cat = X_todo.assign(seller_id=todo["seller_id"].astype(str).values, category_id=todo["category_id"].astype(str).values,
                      palabra1=palabra1.values)
CATS = CAT + ["seller_id", "category_id", "palabra1"]

def categoricas_como_texto(A):
    A = A.copy()
    for c in CATS:
        A[c] = A[c].astype(object).fillna("NA").astype(str)
    return A

PARAMS_CAT = dict(iterations=2000, learning_rate=0.06, depth=6, max_ctr_complexity=2, random_seed=42, verbose=0,
                  thread_count=8, allow_writing_files=False)
inicio = time.time()
p_cat = oof(X_cat, CatBoostClassifier, PARAMS_CAT, preparar=categoricas_como_texto, cat_features=CATS)
print(f"listo en {(time.time() - inicio) / 60:.1f} minutos")
comparar("2. CatBoost con categóricas de muchos valores (vs. LightGBM)", p_cat, p_lgbm)
comparar("2. promedio LightGBM + XGBoost + CatBoost (vs. 05)", (p_lgbm + p_xgb + p_cat) / 3, p_05)
print(f"no coincide con LightGBM en el {desacuerdo(p_cat, p_lgbm):.1%} de los ítems")

listo en 17.5 minutos
2. CatBoost con categóricas de muchos valores (vs. LightGBM) 0.9397  (-0.0046, mejora en 0/5)
2. promedio LightGBM + XGBoost + CatBoost (vs. 05)           0.9447  (+0.0004, mejora en 3/5)
no coincide con LightGBM en el 2.3% de los ítems


**Lectura:** CatBoost queda bastante por debajo de LightGBM, y sumarlo al promedio no mejora de forma consistente. Sus encodings automáticos no le ganan a los que ya están calculados con cuidado (fuera de fold y suavizados), y es el modelo más lento. **Se descarta**, aunque se lo vuelve a probar en el stacking (sección 5) porque se equivoca en lugares distintos.

## 3. SVD del texto

Hoy el texto entra al modelo como **un solo número**: la probabilidad de usado que da la regresión logística sobre el TF-IDF. El TF-IDF tiene unas 400.000 columnas (palabras, pares de palabras y pedazos de letras), casi todas en cero.

La **SVD** (*descomposición en valores singulares*, con `TruncatedSVD`) resume esas 400.000 columnas en `k` columnas numéricas que capturan los **patrones principales** del texto: qué palabras suelen aparecer juntas (ropa, libros, repuestos de auto, celulares...). Con esas columnas, LightGBM puede **cruzar** el tipo de texto con las demás features, en vez de recibir el texto ya resumido en una probabilidad.

`componentes_svd` (en `features.py`) arma el TF-IDF del texto combinado y le aplica la SVD. Se ajusta sobre train + test porque **no usa la respuesta** (como los conteos del 02). `random_state=42` la hace reproducible: calculada dos veces da exactamente lo mismo.

Se prueban varios `k`, cada uno de dos formas: **LightGBM con las columnas SVD solo**, y **promediado con el LightGBM sin SVD**. Tarda unos 20 minutos.

In [6]:
svd = {}
p_svd = {}
for k in [16, 32, 64, 128, 256]:
    inicio = time.time()
    svd[k] = componentes_svd(todo, k)
    p_svd[k] = oof(pd.concat([X_todo, svd[k]], axis=1))
    print(f"k={k} listo en {(time.time() - inicio) / 60:.1f} minutos")
    comparar(f"3. LightGBM con SVD {k}, solo (vs. LightGBM)", p_svd[k], p_lgbm)
    comparar(f"3. promedio LightGBM + LightGBM con SVD {k} (vs. 05)", (p_lgbm + p_svd[k]) / 2, p_05)

k=16 listo en 2.2 minutos
3. LightGBM con SVD 16, solo (vs. LightGBM)                  0.9446  (+0.0002, mejora en 3/5)
3. promedio LightGBM + LightGBM con SVD 16 (vs. 05)          0.9447  (+0.0003, mejora en 3/5)


k=32 listo en 2.4 minutos
3. LightGBM con SVD 32, solo (vs. LightGBM)                  0.9444  (+0.0000, mejora en 2/5)
3. promedio LightGBM + LightGBM con SVD 32 (vs. 05)          0.9449  (+0.0006, mejora en 5/5)


k=64 listo en 2.9 minutos
3. LightGBM con SVD 64, solo (vs. LightGBM)                  0.9447  (+0.0003, mejora en 3/5)
3. promedio LightGBM + LightGBM con SVD 64 (vs. 05)          0.9452  (+0.0009, mejora en 5/5)


k=128 listo en 4.1 minutos
3. LightGBM con SVD 128, solo (vs. LightGBM)                 0.9452  (+0.0008, mejora en 4/5)
3. promedio LightGBM + LightGBM con SVD 128 (vs. 05)         0.9457  (+0.0014, mejora en 5/5)


k=256 listo en 7.3 minutos
3. LightGBM con SVD 256, solo (vs. LightGBM)                 0.9443  (-0.0001, mejora en 2/5)
3. promedio LightGBM + LightGBM con SVD 256 (vs. 05)         0.9452  (+0.0008, mejora en 5/5)


**Lectura:** este es el resultado del notebook, y vale la pena entenderlo bien:

- **Solo**, el LightGBM con SVD mejora poco o nada al LightGBM sin SVD: con 128 componentes mejora algo, pero no en todas las particiones; con el resto, nada. Las dos formas de ver el texto llevan casi la misma información.
- **Promediados**, mejoran **en las 5 particiones**, y más que el modelo con SVD solo. El modelo con SVD y el modelo sin SVD **se equivocan en ítems distintos**: uno mira el texto como un número, el otro como `k` patrones. Al promediarlos, donde uno duda y el otro está seguro, gana el seguro.
- **`k=128`** es el mejor. Con menos componentes la SVD resume demasiado. Con 256 empieza a capturar detalles que son ruido.

### 3.1 ¿Es la SVD, o es solo "otro modelo"?

Una duda razonable: quizás promediar **cualquier** segundo LightGBM mejora, solo porque tiene otra aleatoriedad (`colsample_bytree=0.5` elige columnas al azar). Para descartarlo se promedia LightGBM con **el mismo modelo con otra semilla**:

In [7]:
p_lgbm_semilla = oof(X_todo, params={**PARAMS_LGBM, "random_state": 1})
comparar("3.1 promedio LightGBM + LightGBM con otra semilla (vs. 05)", (p_lgbm + p_lgbm_semilla) / 2, p_05)
print(f"no coinciden en: otra semilla {desacuerdo(p_lgbm, p_lgbm_semilla):.2%} | con SVD 128 {desacuerdo(p_lgbm, p_svd[128]):.2%}")

3.1 promedio LightGBM + LightGBM con otra semilla (vs. 05)   0.9442  (-0.0001, mejora en 3/5)
no coinciden en: otra semilla 0.57% | con SVD 128 1.48%


**Lectura:** otra semilla **no** mejora nada, y se equivoca casi en los mismos ítems. La SVD sí cambia **dónde** se equivoca el modelo (más del doble de desacuerdo), y eso es lo que aprovecha el promedio. **La mejora es por la SVD, no por el azar.**

### 3.2 ¿Y XGBoost con SVD?

En el 05, el modelo final promediaba LightGBM y XGBoost. Se prueba XGBoost con SVD 128 y los promedios posibles. Tarda unos 13 minutos.

In [8]:
X_svd = pd.concat([X_todo, svd[128]], axis=1)
p_xgb_svd = oof(X_svd, XGBClassifier, PARAMS_XGB)
p_par = (p_lgbm + p_svd[128]) / 2
comparar("3.2 promedio LightGBM + LightGBM SVD (el par, vs. 05)", p_par, p_05)
comparar("3.2 promedio de 4: LGBM y XGB, con y sin SVD (vs. 05)", (p_lgbm + p_xgb + p_svd[128] + p_xgb_svd) / 4, p_05)
comparar("3.2 promedio de 3: LGBM, LGBM SVD, XGB SVD (vs. 05)", (p_lgbm + p_svd[128] + p_xgb_svd) / 3, p_05)

3.2 promedio LightGBM + LightGBM SVD (el par, vs. 05)        0.9457  (+0.0014, mejora en 5/5)
3.2 promedio de 4: LGBM y XGB, con y sin SVD (vs. 05)        0.9456  (+0.0013, mejora en 5/5)
3.2 promedio de 3: LGBM, LGBM SVD, XGB SVD (vs. 05)          0.9457  (+0.0014, mejora en 5/5)


**Lectura:** sumar XGBoost (con o sin SVD) da lo mismo que el par de LightGBM. **XGBoost ya no aporta**, así que el modelo se simplifica: **dos LightGBM**, uno sin SVD y otro con SVD 128. Desde acá, la referencia es **ese par**.

## 4. Otros modelos para el ensamble

### 4.1 Extra Trees, Random Forest y el modelo lineal

Modelos que aprenden de forma **muy distinta** al *boosting*:

- **Random Forest:** muchos árboles profundos, cada uno con una muestra al azar de filas y columnas; se promedian (*bagging*).
- **Extra Trees:** como Random Forest, pero además los cortes de cada árbol se eligen al azar. Todavía más distintos entre sí.
- **Modelo lineal** del 05: regresión logística sobre TF-IDF + una columna 0/1 por vendedor, categoría, tipo de publicación, etc.

Los árboles de `sklearn` necesitan todo numérico: las categóricas pasan a su código y los vacíos a −1. Tarda unos 11 minutos.

In [9]:
def todo_numerico(A):
    A = A.copy()
    for c in CAT:
        A[c] = A[c].cat.codes
    return A.fillna(-1).astype(np.float32)

PARAMS_ARBOLES = dict(n_estimators=500, min_samples_leaf=2, max_features=0.3, n_jobs=8, random_state=42)
p_et = oof(X_todo, ExtraTreesClassifier, PARAMS_ARBOLES, preparar=todo_numerico)
p_rf = oof(X_todo, RandomForestClassifier, PARAMS_ARBOLES, preparar=todo_numerico)

texto = texto_combinado(todo)
vend, cat = todo["seller_id"].astype(str), todo["category_id"].astype(str)
W = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, dtype=np.float32).fit_transform(texto)
C_ = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, sublinear_tf=True, max_features=200_000,
                     dtype=np.float32).fit_transform(texto)
categoricas = pd.DataFrame({
    "vend": vend, "cat": cat, "tipo": todo["listing_type_id"], "vend_tipo": vend + todo["listing_type_id"], "vend_cat": vend + cat,
    "modo": todo["buying_mode"], "envio": todo["shipping"].apply(lambda s: str(g(s, "mode"))),
    "foto": todo["pictures"].apply(lambda L: L[0]["max_size"] if L else "no"),
    "cant": todo["initial_quantity"].clip(upper=10).astype(str),
    "precio": pd.qcut(np.log1p(todo["price"]), 20, labels=False, duplicates="drop").astype(str)})
M = sp.hstack([W, C_, OneHotEncoder(handle_unknown="ignore", dtype=np.float32).fit_transform(categoricas)]).tocsr()
p_lineal = np.zeros(N_TRAIN)
for idx_a, idx_b in PARTICIONES[:5]:
    p_lineal[idx_b] = LogisticRegression(max_iter=3000, C=1).fit(M[idx_a], y[idx_a]).predict_proba(M[idx_b])[:, 1]

for nombre, p in [("Extra Trees", p_et), ("Random Forest", p_rf), ("modelo lineal", p_lineal)]:
    comparar(f"4.1 {nombre}, solo (vs. el par)", p, p_par)
    comparar(f"4.1 par + {nombre} (peso 0,2)", 0.8 * p_par + 0.2 * p, p_par)

4.1 Extra Trees, solo (vs. el par)                           0.9340  (-0.0117, mejora en 0/5)
4.1 par + Extra Trees (peso 0,2)                             0.9456  (-0.0001, mejora en 2/5)
4.1 Random Forest, solo (vs. el par)                         0.9380  (-0.0077, mejora en 0/5)
4.1 par + Random Forest (peso 0,2)                           0.9454  (-0.0003, mejora en 1/5)
4.1 modelo lineal, solo (vs. el par)                         0.9239  (-0.0218, mejora en 0/5)
4.1 par + modelo lineal (peso 0,2)                           0.9454  (-0.0003, mejora en 0/5)


### 4.2 Modelos que ven distintas columnas

Siguiendo la lección de la SVD (un modelo que **mira distinto** se equivoca en otros lugares), se entrenan LightGBM que ven solo una parte de las columnas:

- **Sin vendedor:** sin nada del historial del vendedor (sus *target encodings*, los cruzados con vendedor, los agregados `vend_*` y el conteo de ítems). Juzga solo por el ítem.
- **Sin texto:** sin la probabilidad del texto, pero con la SVD y las palabras clave.
- **Top 40:** solo las 40 features más importantes (según el LightGBM de la partición 0).

Tarda unos 8 minutos.

In [10]:
TODAS = list(X_svd.columns) + list(piezas[0])
VENDEDOR = [c for c in TODAS if c.startswith("vend") or c in ["te_seller_id", "te_vend_cat", "te_vend_tipo", "vendedor_n_items"]]
idx_a, idx_b = PARTICIONES[0]
A0, _ = armar_conjunto(X_todo, piezas[0], idx_a, idx_b)
importancia = pd.Series(LGBMClassifier(**PARAMS_LGBM).fit(A0, y[idx_a]).booster_.feature_importance("gain"), index=A0.columns)
TOP40 = list(importancia.sort_values(ascending=False).index[:40])

p_vistas = {
    "sin vendedor": oof(X_todo, columnas=[c for c in A0.columns if c not in VENDEDOR]),
    "sin texto":    oof(X_svd,  columnas=[c for c in TODAS if c != "prob_combinado"]),
    "top 40":       oof(X_todo, columnas=TOP40),
}
for nombre, p in p_vistas.items():
    comparar(f"4.2 {nombre}, solo (vs. el par)", p, p_par)
    print(f"   no coincide con LightGBM en el {desacuerdo(p, p_lgbm):.1%}")
    for w in [0.15, 0.25]:
        comparar(f"4.2 par + {nombre} (peso {w})", (1 - w) * p_par + w * p, p_par)

4.2 sin vendedor, solo (vs. el par)                          0.9332  (-0.0125, mejora en 0/5)
   no coincide con LightGBM en el 3.3%
4.2 par + sin vendedor (peso 0.15)                           0.9457  (+0.0000, mejora en 2/5)
4.2 par + sin vendedor (peso 0.25)                           0.9458  (+0.0001, mejora en 3/5)
4.2 sin texto, solo (vs. el par)                             0.9408  (-0.0049, mejora en 0/5)
   no coincide con LightGBM en el 2.4%
4.2 par + sin texto (peso 0.15)                              0.9458  (+0.0001, mejora en 2/5)
4.2 par + sin texto (peso 0.25)                              0.9452  (-0.0005, mejora en 1/5)
4.2 top 40, solo (vs. el par)                                0.9428  (-0.0029, mejora en 0/5)
   no coincide con LightGBM en el 1.1%
4.2 par + top 40 (peso 0.15)                                 0.9456  (-0.0001, mejora en 2/5)
4.2 par + top 40 (peso 0.25)                                 0.9454  (-0.0003, mejora en 0/5)


**Lectura de la sección 4:** ninguno suma al par. Son modelos **distintos**, pero **demasiado débiles**: para que un modelo sume a un promedio tiene que equivocarse en otros lugares **y** ser casi tan bueno como los demás. El LightGBM con SVD cumple las dos cosas; estos, solo la primera.

## 5. Stacking

En vez de promediar con pesos fijos, un **modelo de nivel 2** (regresión logística) aprende a combinar las probabilidades de todos los modelos de nivel 1. Recibe el *logit* de cada probabilidad, el tamaño del vendedor y sus productos (así puede aprender, por ejemplo, a confiar más en el modelo "sin vendedor" cuando el vendedor tiene pocas publicaciones).

El nivel 2 se evalúa con las mismas 5 particiones: se entrena con las predicciones *out of fold* de 4 y se prueba en la quinta.

In [11]:
nivel1 = {"lgbm": p_lgbm, "lgbm_svd": p_svd[128], "xgb": p_xgb, "xgb_svd": p_xgb_svd, "catboost": p_cat,
          "extra_trees": p_et, "random_forest": p_rf, "lineal": p_lineal, **p_vistas}
logit = lambda p: np.log(np.clip(p, 1e-5, 1 - 1e-5) / (1 - np.clip(p, 1e-5, 1 - 1e-5)))
log_n_items = np.log1p(X_todo["vendedor_n_items"].iloc[:N_TRAIN].values)

def stacking(nombres, con_tamano=True):
    Z = np.column_stack([logit(nivel1[n]) for n in nombres])
    if con_tamano:
        Z = np.column_stack([Z, log_n_items, Z * log_n_items[:, None]])
    p = np.zeros(N_TRAIN)
    for idx_a, idx_b in PARTICIONES[:5]:
        p[idx_b] = LogisticRegression(C=1, max_iter=3000).fit(Z[idx_a], y[idx_a]).predict_proba(Z[idx_b])[:, 1]
    return p

comparar("5. stacking: lgbm + lgbm_svd", stacking(["lgbm", "lgbm_svd"], con_tamano=False), p_par)
comparar("5. stacking: los 5 boosting + tamaño", stacking(["lgbm", "lgbm_svd", "xgb", "xgb_svd", "catboost"]), p_par)
comparar(f"5. stacking: los {len(nivel1)} modelos + tamaño", stacking(list(nivel1)), p_par)

5. stacking: lgbm + lgbm_svd                                 0.9458  (+0.0001, mejora en 3/5)


5. stacking: los 5 boosting + tamaño                         0.9461  (+0.0004, mejora en 4/5)


5. stacking: los 11 modelos + tamaño                         0.9461  (+0.0004, mejora en 4/5)


**Lectura:** el stacking de todos los modelos queda apenas por encima del par, pero sin mejorar en todas las particiones: está dentro del ruido. Y es muchísimo más complejo: 11 modelos para entrenar, guardar y reproducir. **Se queda el promedio simple del par.**

## 6. Reglas a mano

¿Mejoraría corregir al modelo con reglas evidentes, como "si el título dice *usado*, es usado"? Para cada regla se compara, **en los ítems que cubre**, cuántos acierta la regla y cuántos el modelo:

In [12]:
tr = todo.iloc[:N_TRAIN]
titulo, garantia = tr["title"].str.lower(), tr["warranty"].fillna("").str.lower()
REGLAS = {
    "título dice usado/usada → usado":          (titulo.str.contains(r"\busad[oa]s?\b"), 1),
    "título dice nuevo/nueva → nuevo":          (titulo.str.contains(r"\bnuev[oa]s?\b"), 0),
    "título dice 'sin uso'/'a estrenar' → nuevo": (titulo.str.contains("sin uso|a estrenar"), 0),
    "título dice sellado/en caja → nuevo":      (titulo.str.contains("sellad|en caja"), 0),
    "garantía 'sin garantía' → usado":          (garantia.str.contains("sin garant"), 1),
    "gold_special / gold_pro → nuevo":          (tr["listing_type_id"].isin(["gold_special", "gold_pro"]), 0),
    "tienda oficial → nuevo":                   (tr["official_store_id"].notna(), 0),
    "cantidad inicial > 50 → nuevo":            (tr["initial_quantity"] > 50, 0),
    "libro digital / ebook → nuevo":            (titulo.str.contains("libro digital|ebook|e-book"), 0),
}
prediccion = (p_par > 0.5).astype(int)
filas = []
for nombre, (cubre, clase) in REGLAS.items():
    cubre = cubre.values
    filas.append({"regla": nombre, "ítems": cubre.sum(), "acierto de la regla": (y[cubre] == clase).mean(),
                  "acierto del modelo": (prediccion[cubre] == y[cubre]).mean(),
                  "aciertos que se ganan (o pierden)": int((y[cubre] == clase).sum() - (prediccion[cubre] == y[cubre]).sum())})
pd.DataFrame(filas).round(4)

,regla,ítems,acierto de la regla,acierto del modelo,aciertos que se ganan (o pierden)
0,título dice usado/usada → usado,600,0.9833,0.9850,-1
1,título dice nuevo/nueva → nuevo,3246,0.8078,0.9442,-443
2,título dice 'sin uso'/'a estrenar' → nuevo,176,0.6250,0.8409,-38
3,título dice sellado/en caja → nuevo,382,0.6990,0.9424,-93
4,garantía 'sin garantía' → usado,3814,0.6534,0.9423,-1102
5,gold_special / gold_pro → nuevo,2147,0.9837,0.9981,-31
6,tienda oficial → nuevo,575,0.9704,0.9983,-16
7,cantidad inicial > 50 → nuevo,3103,0.9823,0.9939,-36
8,libro digital / ebook → nuevo,743,0.7820,0.9610,-133


**Lectura:** **todas las reglas empeoran**. El modelo ya aprendió solo lo que dicen esas palabras, y además aprendió **en qué contexto** significan otra cosa. Por ejemplo, "sin uso" es nuevo solo ~62% de las veces: mucha gente publica "sin uso" algo usado que está como nuevo, y lo marca como usado. Una regla fija no puede ver ese contexto.

## 7. Resumen y modelo final

**Modelo 06:** promedio de las probabilidades de **dos LightGBM** con los mismos parámetros:

1. con las 98 columnas del modelo 05 (92 features + 6 piezas);
2. con esas mismas columnas **+ las 128 de la SVD del texto**.

Se predice "usado" si el promedio es mayor a 0,5. XGBoost sale del modelo porque ya no aporta.

In [13]:
comparar("7. MODELO 06: LightGBM + LightGBM con SVD 128 (vs. 05)", p_par, p_05)
pd.DataFrame(resultados)

7. MODELO 06: LightGBM + LightGBM con SVD 128 (vs. 05)       0.9457  (+0.0014, mejora en 5/5)


,experimento,accuracy,vs. referencia,mejora en,diferencia por partición
0,2. CatBoost con categóricas de muchos valores ...,0.9397,-0.0046,0/5,-0.0072 -0.0008 -0.0061 -0.0029 -0.0061
1,2. promedio LightGBM + XGBoost + CatBoost (vs....,0.9447,0.0004,3/5,-0.0009 +0.0007 +0.0013 +0.0018 -0.0009
2,"3. LightGBM con SVD 16, solo (vs. LightGBM)",0.9446,0.0002,3/5,-0.0004 +0.0008 -0.0004 +0.0004 +0.0006
3,3. promedio LightGBM + LightGBM con SVD 16 (vs...,0.9447,0.0003,3/5,-0.0004 +0.0006 +0.0004 +0.0010 +0.0000
4,"3. LightGBM con SVD 32, solo (vs. LightGBM)",0.9444,0.0000,2/5,-0.0002 +0.0003 -0.0002 -0.0002 +0.0005
5,3. promedio LightGBM + LightGBM con SVD 32 (vs...,0.9449,0.0006,5/5,+0.0006 +0.0009 +0.0007 +0.0004 +0.0002
6,"3. LightGBM con SVD 64, solo (vs. LightGBM)",0.9447,0.0003,3/5,-0.0001 +0.0002 +0.0004 -0.0002 +0.0014
7,3. promedio LightGBM + LightGBM con SVD 64 (vs...,0.9452,0.0009,5/5,+0.0001 +0.0006 +0.0011 +0.0015 +0.0010
8,"3. LightGBM con SVD 128, solo (vs. LightGBM)",0.9452,0.0008,4/5,+0.0001 +0.0012 +0.0009 -0.0001 +0.0021
9,3. promedio LightGBM + LightGBM con SVD 128 (v...,0.9457,0.0014,5/5,+0.0010 +0.0011 +0.0019 +0.0017 +0.0011


In [14]:
import os

idx_a, idx_b = PARTICIONES[5]
p_test = np.zeros(len(idx_b))
for X in [X_todo, X_svd]:
    A, B = armar_conjunto(X, piezas[5], idx_a, idx_b)
    p_test += LGBMClassifier(**PARAMS_LGBM).fit(A, y).predict_proba(B)[:, 1] / 2

sub06 = pd.DataFrame({"ID": range(1, len(idx_b) + 1), "condition": np.where(p_test > 0.5, "used", "new")})
os.makedirs("submissions", exist_ok=True)
sub06.to_csv("submissions/06_lgbm_svd.csv", index=False)

anterior = pd.read_csv("submissions/05_encodings_cruzados.csv")
print(sub06["condition"].value_counts(normalize=True).round(3).to_dict())
print(f"cambian {(sub06['condition'] != anterior['condition']).sum()} predicciones respecto de la submission 05")

{'new': 0.529, 'used': 0.471}
cambian 212 predicciones respecto de la submission 05
